In [1]:
from datasets.fsmol_dock import FsDockDataset
from datasets.partitioned_fsmol_dock import FsDockDatasetPartitioned
# from visualize import make_fig



/home/alon.kitin/miniconda3/envs/FSdock/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/alon.kitin/miniconda3/envs/FSdock/lib/python3.9/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [2]:

import torch


def mask_graph_sidechains(graph, molecule_sidechain_mask_idx):
    device = graph['ligand'].x.device
    masks = {
        node_t:(
            (graph.sidechains_mask < molecule_sidechain_mask_idx).to(device)
            if node_t == "ligand"
            else torch.arange(graph[node_t].num_nodes, device=device)
        )
        for node_t in graph.metadata()[0]
    }
    return graph.subgraph(masks)


In [3]:
ds = FsDockDatasetPartitioned('data/fsdock/smol','data/fsdock/smol_tasks.csv', num_workers=torch.get_num_threads(), core_weight=0.7)
ds.load()

[2025-Jul-06 20:18:20 IDT] [partitioned_fsmol_dock.py:250] INFO - started load
[2025-Jul-06 20:18:20 IDT] [partitioned_fsmol_dock.py:254] INFO - started load ligands
[2025-Jul-06 20:18:21 IDT] [partitioned_fsmol_dock.py:257] INFO - started load tasks
[2025-Jul-06 20:18:23 IDT] [partitioned_fsmol_dock.py:260] INFO - started load prots
[2025-Jul-06 20:18:26 IDT] [partitioned_fsmol_dock.py:272] INFO - finished load


In [16]:
import plotly.graph_objects as go

def plotly_edges(graph, start_l, end_l):
    start_pos = graph[start_l].pos
    end_pos = graph[end_l].pos
    edges = graph[start_l, end_l].edge_index
    xe=[]
    ye=[]
    ze=[]
    for s_pos, e_pos in zip(start_pos[edges[0]],end_pos[edges[1]]):
        xe+=[s_pos[0], e_pos[0], None]
        ye+=[s_pos[1], e_pos[1], None]
        ze+=[s_pos[2], e_pos[2], None]
    return {'x': xe, 'y':ye, 'z':ze, 'mode' : 'lines', 'name': f'{start_l}-{end_l}'}


def plotly_molecule_edges(graph):
    x =graph['ligand', 'ligand'].edge_attr.sum(1) !=0
    graph['ligand', 'ligand'].edge_index = graph['ligand', 'ligand'].edge_index[:,x]
    graph['ligand', 'ligand'].edge_attr = graph['ligand', 'ligand'].edge_attr[x,:]
    
    return plotly_edges(graph, 'ligand', 'ligand')
    

def plotly_nodes(graph,l):
    poses = graph[l].pos
    xn=[]
    yn=[]
    zn=[]
    for pos in poses:
        xn.append(pos[0])
        yn.append(pos[1])
        zn.append(pos[2])
    return {'x': xn, 'y':yn, 'z':zn, 'mode': 'markers', 'name': l}

def plotly_holes(graph):
    poses = graph['ligand'].pos
    xn=[]
    yn=[]
    zn=[]
    for pos in poses[graph.hole_neighbors]:
        xn.append(pos[0])
        yn.append(pos[1])
        zn.append(pos[2])
    return {'x': xn, 'y':yn, 'z':zn, 'mode': 'markers', 'name': 'holes'}

def make_fig(graph) -> go.Figure:
    traces=[
        go.Scatter3d(**plotly_molecule_edges(graph), line=dict(color='red', width=4)),
        go.Scatter3d(**plotly_nodes(graph, 'ligand'), marker=dict(symbol='circle', size=5, color='blue')),
        go.Scatter3d(**plotly_nodes(graph, 'receptor'), marker=dict(symbol='circle', size=5, color='green')),
        go.Scatter3d(**plotly_nodes(graph, 'atom'), marker=dict(symbol='circle', size=5, color='orange')),
        go.Scatter3d(**plotly_edges(graph, 'atom', 'receptor'), line=dict(color='brown', width=0.3)),
        go.Scatter3d(**plotly_edges(graph, 'atom', 'atom'), line=dict(color='purple', width=0.3)),
        go.Scatter3d(**plotly_edges(graph, 'receptor', 'receptor'), line=dict(color='grey', width=1)),
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'atom'), line=dict(color='light blue', width=0.3)),
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'receptor'), line=dict(color='green', width=0.3)),
        go.Scatter3d(**plotly_holes(graph), marker=dict(symbol='circle', size=7, color='black')),
        
    ]
    # graph = mask_graph_sidechains(graph, molecule_sidechain_mask_idx=1)
    # traces += [
    #     go.Scatter3d(**plotly_molecule_edges(graph), line=dict(color='red', width=5)),
        
    # ]
    fig = go.Figure(data=traces, layout=go.Layout(
        template='simple_white',
    width=800,
    height=800
        ))
    return fig

In [17]:
graph = ds[0]
make_fig(graph)


In [ ]:
support_mols = ds[[('CHEMBL1119333',1), ('CHEMBL1119333',2),('CHEMBL1119333',3)]]

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

support_mols[0].smiles
Draw.MolsToImage(list(map(lambda x: Chem.MolFromSmiles(x.smiles), support_mols)))

In [ ]:
def make_support_fig(graphs) -> go.Figure:
    colors = ['blue', 'orange', 'purple', ]
    traces2=[[
        go.Scatter3d(**plotly_molecule_edges(graph), line=dict(color=c, width=5)),
        go.Scatter3d(**plotly_nodes(graph, 'receptor'), marker=dict(symbol='circle', size=3, color='green')),
        go.Scatter3d(**plotly_edges(graph, 'receptor', 'receptor'), line=dict(color='grey', width=1)),
        # go.Scatter3d(**plotly_edges(graph, 'ligand', 'atom'), line=dict(color='light blue', width=3)),
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'receptor'), line=dict(color='green', width=0.5)),
        
    ] for c, graph in zip(colors, graphs)]
    traces = sum(traces2, [])
    fig = go.Figure(data=traces, layout=go.Layout(
        template='simple_white',
    width=1000,
    height=1000
        ))
    return fig

In [ ]:
make_support_fig(support_mols)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

sc = '''[1*]CC.[2*]NCC(C)(C)O.[3*]F.[4*]F
[1*]CCN(C)CC[2*]
[1*]C.[2*]C=CCCc1ccco1.[3*]C
[1*]OC.[2*]O.[3*]OC.[4*]OC.[5*]C
[1*]OC.[2*]C.[3*]C(=O)Nc1ccccn1
[1*]OC.[2*]C.[3*]OC
[1*]CC.[2*]F.[3*]F.[4*]C#Cc1ccc(N)nc1
[1*]C(=O)OCC.[2*]OC.[3*]C
[1*]c1ccccc1C(C)=O
[1*]c1ccc([N+](=O)[O-])cc1.[2*]F
[1*]c1cc2c(=O)[nH][nH]c2nn1
[1*]C(=O)NCC(C)(C)C(=O)O.[2*]C#N.[3*]O
[1*]CO.[2*]O.[3*]C(F)(F)F.[4*]O
[1*]C.[2*]C.[3*]C.[4*]C.[5*]C(C)C
[1*]C(=O)Nc1ncc(C)s1.[2*]F.[3*]C
[1*]C
[1*]C.[2*]C.[3*]O.[4*]C.[5*]C
[1*]C=CC(=O)NC.[2*]N.[3*]F.[4*]C
[1*]C(C)C.[2*]CC#N.[3*]N
[1*]C(C)C.[2*]NCCCO
[1*]C.[2*]Cc1ccccc1
[1*]C#N.[2*]CC(=O)O.[3*]Cl.[4*]O.[5*]Oc1ccccc1
[1*]C.[2*]S(=O)(=O)C=CC#N
[1*]OCOC.[2*]OCc1ccccc1.[3*]c1cc(=O)c2cc(F)ccc2o1
[1*]C(=O)c1cnccn1.[2*]c1ccccc1'''.split('\n')
# Draw.MolsToImage(list(map(lambda x: Chem.MolFromSmiles(x), sc)))
Draw.MolsToGridImage(list(map(lambda x: Chem.MolFromSmiles(x), sc)),
                        molsPerRow=4, subImgSize=(400,400))